# Lesson 18 — Observability & Monitoring

## What you will learn
- **Structured JSON logging** for every node execution
- **Metrics collection**: latency, token counts, error rates
- **Tracing** with LangSmith (optional) and manual trace IDs
- How to use LangGraph's `stream_mode="debug"` for deep inspection
- **Health check** endpoint pattern for ALB/load balancers

## Observability pillars
```
┌────────────┬────────────────────────────────────────────────┐
│ Pillar     │ Implementation                                  │
├────────────┼────────────────────────────────────────────────┤
│ Logging    │ structlog / JSON to CloudWatch                  │
│ Metrics    │ Latency per node, token cost, error rate        │
│ Tracing    │ trace_id → propagated through all nodes         │
│ Alerting   │ error_count > threshold → SNS/PagerDuty         │
└────────────┴────────────────────────────────────────────────┘
```

In [ ]:
# !pip install langgraph langchain-ollama

## Step 1 — Structured JSON Logger

In [ ]:
import json
import time
import uuid
import logging
from typing import Annotated
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

class JsonFormatter(logging.Formatter):
    """Emit every log record as a JSON line."""
    def format(self, record: logging.LogRecord) -> str:
        log = {
            "ts":      self.formatTime(record, "%Y-%m-%dT%H:%M:%S"),
            "level":   record.levelname,
            "logger":  record.name,
            "message": record.getMessage(),
        }
        if hasattr(record, "extra"):
            log.update(record.extra)
        return json.dumps(log)

handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
logger = logging.getLogger("langgraph.obs")
logger.handlers = [handler]
logger.setLevel(logging.INFO)
logger.propagate = False

def log_event(event: str, **kwargs):
    record = logging.LogRecord(
        name="langgraph.obs", level=logging.INFO,
        pathname="", lineno=0, msg=event, args=(), exc_info=None
    )
    record.extra = kwargs
    logger.handle(record)

log_event("logger_ready", service="langgraph-demo", version="1.0.0")

## Step 2 — State with Observability Fields

In [ ]:
def merge_metrics(existing: dict, new: dict) -> dict:
    merged = dict(existing)
    for k, v in new.items():
        if isinstance(v, (int, float)) and k in merged:
            merged[k] = merged[k] + v
        else:
            merged[k] = v
    return merged

class ObsState(TypedDict):
    messages:   Annotated[list, add_messages]
    trace_id:   str           # unique per request
    tenant_id:  str
    metrics:    Annotated[dict, merge_metrics]  # accumulated
    error:      str

print("State defined with trace_id + metrics accumulation")

## Step 3 — Observable Node Wrapper

In [ ]:
def observable(node_name: str):
    """Decorator: wraps any node function with structured logging + latency tracking."""
    def decorator(fn):
        def wrapper(state):
            t0 = time.perf_counter()
            log_event("node_start",
                node=node_name,
                trace_id=state.get("trace_id", ""),
                tenant_id=state.get("tenant_id", ""),
                msg_count=len(state.get("messages", []))
            )
            try:
                result = fn(state)
                elapsed_ms = (time.perf_counter() - t0) * 1000
                log_event("node_end",
                    node=node_name,
                    trace_id=state.get("trace_id", ""),
                    latency_ms=round(elapsed_ms, 1),
                    status="ok"
                )
                # Accumulate latency metric
                if result is None:
                    result = {}
                result.setdefault("metrics", {})
                result["metrics"][f"{node_name}_latency_ms"] = round(elapsed_ms, 1)
                return result
            except Exception as e:
                elapsed_ms = (time.perf_counter() - t0) * 1000
                log_event("node_error",
                    node=node_name,
                    trace_id=state.get("trace_id", ""),
                    error=str(e),
                    latency_ms=round(elapsed_ms, 1),
                    status="error"
                )
                return {"error": str(e), "metrics": {f"{node_name}_errors": 1}}
        return wrapper
    return decorator

print("@observable decorator ready")

## Step 4 — Build Observable Agent

In [ ]:
llm = ChatOllama(model="llama3", temperature=0)

@observable("preprocess")
def preprocess_node(state: ObsState) -> dict:
    """Validate and normalize the input."""
    last = state["messages"][-1].content if state["messages"] else ""
    return {"messages": [HumanMessage(content=last.strip())]}

@observable("llm_call")
def llm_node(state: ObsState) -> dict:
    """Call LLM."""
    system = SystemMessage(content="You are a helpful assistant. Be concise.")
    response = llm.invoke([system] + state["messages"])
    # Track approximate token count
    token_estimate = len(response.content.split())
    return {"messages": [response], "metrics": {"tokens_out": token_estimate}}

@observable("postprocess")
def postprocess_node(state: ObsState) -> dict:
    """Format the final output."""
    return {}

builder = StateGraph(ObsState)
builder.add_node("preprocess",  preprocess_node)
builder.add_node("llm_call",    llm_node)
builder.add_node("postprocess", postprocess_node)
builder.add_edge(START,          "preprocess")
builder.add_edge("preprocess",   "llm_call")
builder.add_edge("llm_call",     "postprocess")
builder.add_edge("postprocess",  END)

graph = builder.compile(checkpointer=MemorySaver())
print("Observable agent compiled!")

## Step 5 — Run and Inspect Metrics

In [ ]:
def run_with_trace(question: str, tenant_id: str = "demo"):
    trace_id = str(uuid.uuid4())[:8]
    config = {"configurable": {"thread_id": f"{tenant_id}:{trace_id}"}}
    
    log_event("request_start", trace_id=trace_id, tenant_id=tenant_id, question=question[:60])
    t0 = time.perf_counter()
    
    result = graph.invoke({
        "messages": [HumanMessage(content=question)],
        "trace_id": trace_id,
        "tenant_id": tenant_id,
        "metrics": {},
        "error": "",
    }, config=config)
    
    total_ms = (time.perf_counter() - t0) * 1000
    log_event("request_end", trace_id=trace_id, total_ms=round(total_ms, 1), metrics=result.get("metrics", {}))
    
    print(f"\nAnswer: {result['messages'][-1].content[:150]}")
    print(f"Metrics: {result.get('metrics', {})}")
    return result

run_with_trace("What is LangGraph?", tenant_id="acme")
run_with_trace("What is 2 + 2?",     tenant_id="beta")

## Step 6 — stream_mode="debug" for Deep Inspection

In [ ]:
config = {"configurable": {"thread_id": "debug-demo"}}
print("Debug stream (node-by-node state updates):")
for chunk in graph.stream(
    {"messages": [HumanMessage(content="Hello!")],
     "trace_id": "debug-001", "tenant_id": "demo", "metrics": {}, "error": ""},
    config=config,
    stream_mode="updates",
):
    for node, update in chunk.items():
        keys = list(update.keys())
        print(f"  [{node}] updated keys: {keys}")

## Key Takeaways

| Pattern | Implementation |
|---------|---------------|
| JSON logs | `JsonFormatter` → stdout → CloudWatch agent |
| Trace ID | Generate `uuid4()` per request → pass in state |
| Node latency | `@observable` decorator wraps any node function |
| Metric accumulation | `Annotated[dict, merge_metrics]` reducer |
| Debug streaming | `graph.stream(..., stream_mode="updates")` |

## 🏋️ Exercise
1. Add an `error_rate_node` at the end that checks `metrics["errors"]` and logs a warning if > 0
2. Add a `p95_latency` tracker: store the last 100 `llm_call_latency_ms` values and compute p95
3. Integrate with LangSmith: set `LANGCHAIN_TRACING_V2=true` and `LANGCHAIN_API_KEY` in `.env` and observe traces in the LangSmith dashboard

In [ ]:
# Your exercise solution here
